# Notebook 1 — Time Grid & Load Profile

This notebook introduces the two foundational building blocks of the simulator:

1. **`TimeGrid`** — the temporal backbone that maps every quarter-hour of the year to calendar metadata (month, hour, day of week, holidays).
2. **`LoadProfile`** — a multiplicative demand model that composes monthly, hourly, weekday, holiday, and stochastic noise factors into a realistic full-year electricity demand curve.

By the end you will understand:
- Why quarter-hour resolution matters for power system simulation
- How the load profile shapes demand across seasons, hours, and weekdays
- The effect of noise on dispatch volatility

**Runtime**: < 10 seconds

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from energy_sim.models import TimeGrid, LoadProfile
from energy_sim.config import (
    QUARTERS_PER_YEAR, QUARTERS_PER_DAY, P_PEAK_GW,
    MONTHLY_LOAD_FACTORS, HOURLY_LOAD_FACTORS,
    WEEKDAY_LOAD_FACTORS, HOLIDAY_LOAD_FACTOR,
    ITALIAN_HOLIDAYS_DOY, DEFAULT_LOAD_NOISE_SIGMA,
)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. The TimeGrid

The simulator runs at **quarter-hour resolution** (15-minute intervals), matching the imbalance settlement period used in most European electricity markets. One simulated year has:

$$T = 365 \times 96 = 35\,040 \text{ quarter-hours}$$

The `TimeGrid` pre-computes vectorized arrays mapping each index to its calendar attributes — no datetime parsing at simulation time.

In [ ]:
tg = TimeGrid()

print(f"Total quarter-hours per year: {tg.n:,}")
print(f"Quarter-hours per day:        {QUARTERS_PER_DAY}")
print(f"\nFirst 10 indices:  {tg.quarter_index[:10]}")
print(f"Corresponding hours: {tg.hour[:10]}")
print(f"Corresponding months: {tg.month[:10]}")
print(f"Day of week (0=Mon): {tg.day_of_week[:10]}")

### How months are distributed

Each month spans a different number of quarter-hours (proportional to its number of days). Let's verify:

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Quarter-hours per month
qh_per_month = [np.sum(tg.month == m) for m in range(1, 13)]
axes[0].bar(month_names, qh_per_month, color='steelblue')
axes[0].set_ylabel('Quarter-hours')
axes[0].set_title('Quarter-hours per month')

# Day-of-week distribution
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
qh_per_dow = [np.sum(tg.day_of_week == d) for d in range(7)]
axes[1].bar(dow_names, qh_per_dow, color='coral')
axes[1].set_ylabel('Quarter-hours')
axes[1].set_title('Quarter-hours per day of week')

plt.tight_layout()
plt.show()

### Holidays

The simulator includes the 10 fixed-date Italian public holidays. Easter is excluded because the simulated year is generic (no specific calendar year).

In [ ]:
tg.set_holiday_calendar(ITALIAN_HOLIDAYS_DOY)

holiday_names = [
    'New Year', 'Epiphany', 'Liberation Day', 'Labour Day',
    'Republic Day', 'Ferragosto', 'All Saints', 'Imm. Conception',
    'Christmas', 'St. Stephen'
]
for doy, name in zip(ITALIAN_HOLIDAYS_DOY, holiday_names):
    month_of_holiday = tg.month[doy * QUARTERS_PER_DAY]
    print(f"  Day {doy:3d} ({month_names[month_of_holiday-1]:>3s}) — {name}")

print(f"\nTotal holiday quarter-hours: {tg.is_holiday.sum()} "
      f"({tg.is_holiday.sum() / QUARTERS_PER_DAY:.0f} days)")

## 2. The LoadProfile

Electricity demand is shaped by a multiplicative model:

$$P(t) = P_{\text{peak}} \times k_{\text{month}}(t) \times k_{\text{hour}}(t) \times k_{\text{weekday}}(t) \times k_{\text{holiday}}(t) \times \epsilon(t)$$

Where each $k$ is a shaping factor and $\epsilon$ is optional Gaussian noise.

### 2.1 Monthly factors

July is the peak month (factor 1.0) due to air conditioning. Spring is the trough.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly factors
months = list(range(1, 13))
m_factors = [MONTHLY_LOAD_FACTORS[m] for m in months]
axes[0].bar(month_names, m_factors, color='steelblue')
axes[0].set_ylabel('Load factor')
axes[0].set_title('Monthly load factors (1.0 = July peak)')
axes[0].set_ylim(0.5, 1.1)
axes[0].axhline(1.0, color='gray', ls='--', alpha=0.5)

# Hourly factors
hours = list(range(24))
h_factors = [HOURLY_LOAD_FACTORS[h] for h in hours]
axes[1].plot(hours, h_factors, 'o-', color='coral', lw=2)
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Load factor')
axes[1].set_title('Hourly load factors (1.0 = evening peak 19:00)')
axes[1].set_ylim(0.4, 1.1)
axes[1].axhline(1.0, color='gray', ls='--', alpha=0.5)
axes[1].set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.show()

### 2.2 Weekday and holiday factors

Industrial and commercial loads drop on weekends and holidays:

In [ ]:
print("Weekday factors:")
for d, name in enumerate(dow_names):
    print(f"  {name}: {WEEKDAY_LOAD_FACTORS[d]:.2f}")

print(f"\nHoliday factor: {HOLIDAY_LOAD_FACTOR:.2f}")
print(f"  → A holiday on Sunday: {WEEKDAY_LOAD_FACTORS[6] * HOLIDAY_LOAD_FACTOR:.2f} of peak")
print(f"    (near-total shutdown of industrial loads)")

### 2.3 Building the full profile

Let's generate a deterministic profile (no noise) and examine its shape:

In [ ]:
lp = LoadProfile(tg, p_peak_pu=1.0)
lp.set_weekday_factors(WEEKDAY_LOAD_FACTORS)
lp.set_holiday_factor(HOLIDAY_LOAD_FACTOR)

# Deterministic profile
load_det = lp.generate()

# Convert index to day-of-year for plotting
days = tg.quarter_index / QUARTERS_PER_DAY

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(days, load_det * P_PEAK_GW, lw=0.3, color='steelblue', alpha=0.7)
ax.set_xlabel('Day of year')
ax.set_ylabel('Load (GW)')
ax.set_title('Full-year load profile (deterministic, no noise)')

# Mark months
cum_days = np.cumsum([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
for i, name in enumerate(month_names):
    ax.axvline(cum_days[i], color='gray', alpha=0.2, ls='-')
    ax.text(cum_days[i] + 15, ax.get_ylim()[1] * 0.95, name,
            ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print(f"Peak load:    {load_det.max() * P_PEAK_GW:.1f} GW")
print(f"Minimum load: {load_det.min() * P_PEAK_GW:.1f} GW")
print(f"Average load: {load_det.mean() * P_PEAK_GW:.1f} GW")

### 2.4 Zooming in: summer vs winter daily shapes

Let's compare a typical week in January vs July:

In [ ]:
# Week in January (days 7-13) and July (days 182-188)
jan_start = 7 * QUARTERS_PER_DAY
jul_start = 182 * QUARTERS_PER_DAY
week = 7 * QUARTERS_PER_DAY
hours_in_week = np.arange(week) / 4  # quarter-hours to hours

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hours_in_week, load_det[jan_start:jan_start+week] * P_PEAK_GW,
        label='January week', color='steelblue', lw=1.5)
ax.plot(hours_in_week, load_det[jul_start:jul_start+week] * P_PEAK_GW,
        label='July week', color='coral', lw=1.5)

# Mark days
for d in range(8):
    ax.axvline(d * 24, color='gray', alpha=0.2)
    if d < 7:
        ax.text(d * 24 + 12, ax.get_ylim()[0] + 1,
                dow_names[(tg.day_of_week[jan_start] + d) % 7],
                ha='center', fontsize=8)

ax.set_xlabel('Hours into the week')
ax.set_ylabel('Load (GW)')
ax.set_title('Daily load shapes: January vs July')
ax.legend()
plt.tight_layout()
plt.show()

Notice:
- July has **higher peak** (air conditioning) and a broader daytime plateau
- Both show the characteristic **double hump** (morning ramp + evening peak)
- **Weekend dips** are clearly visible (Saturday at 85%, Sunday at 75%)

## 3. The effect of noise

Real demand is not perfectly predictable — weather-driven HVAC swings, random industrial loads, etc. The `noise_sigma` parameter adds multiplicative Gaussian noise:

$$P_{\text{noisy}}(t) = P_{\text{det}}(t) \times \mathcal{N}(1, \sigma^2), \quad \text{clipped to } [0.5, 1.5]$$

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, sigma in enumerate([0.0, 0.02, 0.04]):
    rng_i = np.random.default_rng(42)
    load_i = lp.generate(rng_i, noise_sigma=sigma)
    
    # Plot one week in July
    axes[i].plot(hours_in_week, load_i[jul_start:jul_start+week] * P_PEAK_GW,
                 lw=0.8, color='coral')
    axes[i].set_title(f'noise_sigma = {sigma}')
    axes[i].set_xlabel('Hours')
    axes[i].set_ylabel('Load (GW)')
    axes[i].set_ylim(20, 62)

plt.suptitle('Effect of load noise on a July week', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Default noise_sigma: {DEFAULT_LOAD_NOISE_SIGMA}")

## 4. Play with parameters

Try modifying the parameters below and re-running to see the impact:

In [ ]:
# ── PLAY WITH THESE ──────────────────────────────────────
custom_noise_sigma = 0.06        # try 0.0, 0.02, 0.04, 0.08
custom_weekend_factor = 0.60     # try 0.5 (deep drop) to 1.0 (no effect)
custom_holiday_factor = 0.50     # try 0.3 (total shutdown) to 1.0 (no effect)
# ─────────────────────────────────────────────────────────

tg2 = TimeGrid()
tg2.set_holiday_calendar(ITALIAN_HOLIDAYS_DOY)

lp2 = LoadProfile(tg2)
custom_weekday = {d: (1.0 if d < 5 else custom_weekend_factor) for d in range(7)}
lp2.set_weekday_factors(custom_weekday)
lp2.set_holiday_factor(custom_holiday_factor)

rng2 = np.random.default_rng(42)
load_custom = lp2.generate(rng2, noise_sigma=custom_noise_sigma)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(days, load_custom * P_PEAK_GW, lw=0.3, color='darkorange', alpha=0.7)
ax.plot(days, load_det * P_PEAK_GW, lw=0.3, color='steelblue', alpha=0.4, label='default')
ax.set_xlabel('Day of year')
ax.set_ylabel('Load (GW)')
ax.set_title(f'Custom profile (noise={custom_noise_sigma}, weekend={custom_weekend_factor}, holiday={custom_holiday_factor})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Peak: {load_custom.max() * P_PEAK_GW:.1f} GW  |  "
      f"Min: {load_custom.min() * P_PEAK_GW:.1f} GW  |  "
      f"Avg: {load_custom.mean() * P_PEAK_GW:.1f} GW")

## Key Takeaways

1. **Quarter-hour resolution** (35,040 steps) captures both slow seasonal trends and fast intra-day dynamics.
2. The **multiplicative model** decomposes demand into independent factors — easy to calibrate and interpret.
3. **Monthly + hourly** factors create the characteristic double seasonality: summer peak + evening peak.
4. **Weekday/holiday** factors model the industrial load cycle — critical for weekend renewables surplus.
5. **Noise** represents forecast error and random variability — even small sigma (0.04) significantly affects dispatch outcomes.

**Next notebook**: [02 — Fuel & Carbon Prices](./02_fuel_and_carbon_prices.ipynb) — how stochastic price processes drive generator economics.